# Calibration gains correlated across channels

Spectro-interferometric data have many wavelength channels per baseline, but their calibration errors are not independent between channels. The transfer function, injection losses and piston drifts multiply every channel of a frame's baseline by much the same factor. Error bars that treat each channel as independent therefore make the data look far more precise than they are, and inflating them uniformly does not fix that.

virgil models such errors as **gains on log |V|**, one per frame and telescope and one per frame and baseline, and integrates them out analytically (`OIData.with_gains`, Stage 6d). This notebook simulates four-telescope K-band data with injected gains and fits them three ways:

1. with the reported errors only;
2. with an extra relative error fitted (`vis_error_rel`), the usual remedy;
3. with the gains marginalised and their widths fitted.

One realisation shows the fits; a small Monte Carlo then checks whether each fit's error bars are right.

The scene is a star, a faint companion and a partly resolved Gaussian disk around the star. The coverage is the four UTs at eight hour angles, in 30 K-band channels. `vlti_oidata` gives each snapshot a frame and each baseline its station pair, which the gains need (data read from OIFITS have them too).

In [ ]:
import jax
import numpy as np
import numpyro.distributions as dist
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from virgil.coverage import vlti_oidata
from virgil.fitting import fit
from virgil.inference import laplace_covariance
from virgil.likelihood import loglike, whitened_residuals
from virgil.models import GaussianDisk, PointSource, System

jax.config.update("jax_enable_x64", True)

TRUTH = System(
    star=PointSource(),
    comp=PointSource(flux=0.1, dra=6.0, ddec=-4.0),
    disk=GaussianDisk(sigma=1.5, flux=0.4),
)
PRIORS = {
    "comp.flux": dist.Uniform(0.0, 1.0),
    "comp.dra": dist.Uniform(-20.0, 20.0),
    "comp.ddec": dist.Uniform(-20.0, 20.0),
    "disk.flux": dist.Uniform(0.0, 2.0),
    "disk.sigma": dist.Uniform(0.05, 10.0),
}
PATHS = list(PRIORS)
TRUE_VALUES = np.array([float(TRUTH.get(p)) for p in PATHS])

template = vlti_oidata(
    hour_angles_h=np.linspace(-3.0, 3.0, 8),
    wavelengths_m=np.linspace(2.0e-6, 2.45e-6, 30),
    sigma_v2=0.01,
    sigma_cp_deg=0.5,
)
# Gains on log |V|: 2% per (frame, telescope) and 1% per (frame, baseline),
# i.e. 4% and 2% on V². with_model draws them along with the noise.
template = template.with_gains(telescope=0.02, baseline=0.01)
data = template.with_model(TRUTH, jax.random.PRNGKey(0))
print(f"{data.vis.size} V², {data.phi.size} closure phases, "
      f"{data.gains.rows.shape[0]} frames of gains, groups {data.gains.groups}")

What the gains do to the data: the ratio of observed to true V² for every channel of the first frame. Within a baseline the channels share one offset (up to the 1% white noise), and baselines through the same telescope share part of it. A model that treats the 30 channels as independent sees each offset 30 times.

In [ ]:
n_wave = 30
ratio = (np.asarray(data.vis) / np.asarray(data.with_model(TRUTH).vis)).reshape(-1, n_wave)
wavel_um = np.linspace(2.0, 2.45, n_wave)
stations = np.asarray(data.stations)[::n_wave]

fig, ax = plt.subplots(figsize=(7, 4))
for row in range(6):  # the six baselines of frame 0
    a, b = stations[row]
    ax.plot(wavel_um, ratio[row], ".-", label=f"UT{a}–UT{b}")
ax.axhline(1.0, color="k", lw=0.8)
ax.set(xlabel="wavelength (µm)", ylabel="V² observed / true", title="Frame 0")
ax.legend(ncol=3, fontsize=8)
plt.show()

Now the three fits of this realisation. Each starts from the truth (the point is the error bars, not the search) and gets Laplace errors at its maximum: the inverse Hessian of the negative log likelihood, with any fitted error terms held at their best values. With gains the covariance depends on the model (through V²), so `fit` uses L-BFGS on the full likelihood, log-determinant included.

In [ ]:
def laplace_errors(data, values, noise):
    x = np.array([values[p] for p in PATHS])

    def objective(v):
        return -loglike(v, PATHS, data, TRUTH, **noise)

    return np.sqrt(np.diag(np.asarray(laplace_covariance(objective, x))))


def without_gains(data):
    import equinox as eqx

    return eqx.tree_at(lambda d: d.gains, data, None, is_leaf=lambda x: x is None)


METHODS = {
    "reported errors": (without_gains, None),
    "+ vis_error_rel": (without_gains, {"vis_error_rel": dist.Uniform(0.0, 0.5)}),
    "gains, widths fitted": (
        lambda d: d,
        {"vis_gain_telescope": dist.Uniform(0.0, 0.3),
         "vis_gain_baseline": dist.Uniform(0.0, 0.3)},
    ),
}


def fit_all(data):
    out = {}
    for name, (prepare, noise_priors) in METHODS.items():
        d = prepare(data)
        result = fit(TRUTH, PRIORS, d, noise=noise_priors)
        noise = {k.removeprefix("noise."): float(v)
                 for k, v in result.values.items() if k.startswith("noise.")}
        values = np.array([float(result.values[p]) for p in PATHS])
        errors = laplace_errors(d, dict(zip(PATHS, values)), noise)
        out[name] = dict(values=values, errors=errors, noise=noise,
                         chi2_red=sum(result.info["chi2"]) / sum(result.info["ndata"]))
    return out


one = fit_all(data)
print(f"{'parameter':<12}{'truth':>8}" + "".join(f"{m:>26}" for m in METHODS))
for i, p in enumerate(PATHS):
    row = "".join(f"{one[m]['values'][i]:>14.4f} ± {one[m]['errors'][i]:<9.4f}" for m in METHODS)
    print(f"{p:<12}{TRUE_VALUES[i]:>8.3f}{row}")
for m in METHODS:
    print(f"{m}: χ²/N = {one[m]['chi2_red']:.2f}, fitted terms {one[m]['noise']}")

With the reported errors, χ²/N is far above 1 and the error bars are tiny. Fitting `vis_error_rel` brings χ²/N to 1, which looks reassuring. The fitted gain widths should come out near the injected 0.02 and 0.01. A single realisation cannot say whose error bars are right, though; for that we repeat the experiment.

A quick check first: with gains, the whitened residuals of the fit should look like independent unit Gaussians, and without them they should not.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
bins = np.linspace(-6, 6, 61)
best = {m: TRUTH.set(PATHS, list(one[m]["values"])) for m in METHODS}
n_vis = data.vis.size
r_diag = np.asarray(whitened_residuals(best["reported errors"], without_gains(data)))[:n_vis]
r_gain = np.asarray(whitened_residuals(
    best["gains, widths fitted"], data,
    **{k: one["gains, widths fitted"]["noise"][k] for k in ("vis_gain_telescope", "vis_gain_baseline")},
))[:n_vis]
ax.hist(r_diag, bins, histtype="step", density=True, label=f"reported errors (rms {r_diag.std():.1f})")
ax.hist(r_gain, bins, histtype="step", density=True, label=f"gains marginalised (rms {r_gain.std():.2f})")
x = np.linspace(-6, 6, 200)
ax.plot(x, np.exp(-x**2 / 2) / np.sqrt(2 * np.pi), "k--", lw=0.8, label="N(0, 1)")
ax.set(xlabel="whitened V² residual", ylabel="density")
ax.legend(fontsize=8)
plt.show()

Now a small Monte Carlo: 40 realisations of the same observation, each with new gains and noise, fitted the three ways. For calibrated errors the **pulls** (fit − truth)/σ have unit standard deviation. (OzSTAR job `stage6d_gains` ran 200 realisations of this setup, with a control without gains; the results are in `design/imaging_plan.md`, Stage 6d.)

In [ ]:
N_REAL = 40
runs = [fit_all(template.with_model(TRUTH, jax.random.PRNGKey(100 + r)))
        for r in tqdm(range(N_REAL), desc="realisations")]

pulls = {m: np.array([(run[m]["values"] - TRUE_VALUES) / run[m]["errors"] for run in runs])
         for m in METHODS}
print(f"{'rms pull':<12}" + "".join(f"{m:>24}" for m in METHODS))
for i, p in enumerate(PATHS):
    print(f"{p:<12}" + "".join(f"{np.sqrt(np.mean(pulls[m][:, i] ** 2)):>24.2f}" for m in METHODS))
widths = np.array([[run["gains, widths fitted"]["noise"][k] for k in ("vis_gain_telescope", "vis_gain_baseline")]
                   for run in runs])
print(f"fitted widths: telescope {widths[:, 0].mean():.4f} ± {widths[:, 0].std():.4f} (true 0.02), "
      f"baseline {widths[:, 1].mean():.4f} ± {widths[:, 1].std():.4f} (true 0.01)")

In [ ]:
fig, axes = plt.subplots(1, len(METHODS), figsize=(12, 3.5), sharey=True)
bins = np.linspace(-6, 6, 25)
x = np.linspace(-6, 6, 200)
for ax, m in zip(axes, METHODS):
    flat = np.clip(pulls[m].ravel(), -6, 6)
    ax.hist(flat, bins, density=True, color="C0", alpha=0.7)
    ax.plot(x, np.exp(-x**2 / 2) / np.sqrt(2 * np.pi), "k--", lw=0.8)
    ax.set(title=f"{m}\nrms pull {np.sqrt(np.mean(pulls[m] ** 2)):.1f}", xlabel="(fit − truth) / σ")
axes[0].set_ylabel("density (all parameters; clipped at ±6)")
plt.tight_layout()
plt.show()

**What this shows.**

- With the reported errors alone, the error bars are an order of magnitude too small: the pulls are spread over many σ.
- Fitting `vis_error_rel` makes χ²/N look right but leaves the error bars several times too small. A larger independent error per channel still averages down over 30 channels, while the gains do not. It also spends the extra error on every channel equally, so the estimates scatter more than they need to.
- With the gains marginalised, the pulls are unit Gaussians and the fitted widths recover the injected ones. The estimates are also more precise than with `vis_error_rel`, because the model knows which part of the scatter is common to a frame's channels.

**In practice.** Read the data from OIFITS (which gives frames and station pairs), add `data.with_gains(telescope=..., baseline=...)` with nominal widths, and fit the widths with `noise={"vis_gain_telescope": ..., "vis_gain_baseline": ...}`. If the data carry no such gains, the fitted widths go to zero and nothing is lost but a little speed. Related tools:

- `with_gains(chromatic=...)` adds a gain shaped (λ_ref/λ)², a coherence loss.
- `with_gains(modes=...)` takes supplied modes, for example a calibrator PCA's templates from virgil-vlti.
- `OIData.with_closure_offsets` does the same for closure phases that do not close (off by default).
- The noise terms `wavel_scale` and `wavel_offset` fit a wavelength calibration error.